# Lab: harden your MCP server, with an agent in the loop

A real agent (DeepSeek, through OpenRouter) uses your music server while the network misbehaves: **one read in four is rate-limited**, and **the first of every three writes is executed but its response is lost**. You watch it fail, fix the **server** (`my_mcp_server.py`) and the **dispatcher** (retry only when safe), and watch it again.

**Setup:** put `OPENROUTER_API_KEY=...` in a `.env` file (see `.env.example`); never commit it. Run the cells in order, **before** editing the server for the first time.

In [ ]:
import hashlib, json
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_random_exponential
from lab import check, trials, typed_error   # given: runs the agent against your server (see lab.py if curious)

## 1. Watch the agent on the given server (v0.1)

Three runs of the same task. The table is what the server **really did**; "agent says" is what the user is told. The model is not deterministic: runs differ.

In [ ]:
before = await trials()
before

## 2. Harden the server

Open `my_mcp_server.py` and do the four TODOs, **below** the simulated service:

1. **annotations**: which tools are read-only, which write is safe to repeat;
2. **constrained arguments**: `Annotated[str, Field(pattern=..., description=...)]`, `Literal[...]`;
3. **typed errors**: `raise tool_error(code, message, retryable=..., outcome_unknown=...)`. A lost write response is `timeout`, retryable, `outcome_unknown=True`;
4. **`idempotency_key: str | None = None`** on the write tool, passed to `svc.add_to_playlist`. The model never fills it in.

Check that it starts: `uv run my_mcp_server.py` (Ctrl+C to stop).

## 3. Harden the dispatcher

The **harness** computes the key from `(run_id, tool, arguments)`, once per logical call. Retry only if the error is `retryable` **and** repeating is safe (read-only, or keyed), with backoff and jitter. Use `tenacity`: do not write the retry loop yourself.

In [ ]:
def idempotency_key(run_id, tool, args):
    # TODO 1: same logical call -> same key, at every attempt. Hash run_id, tool and the sorted arguments.
    # No timestamp, no random value.
    raise NotImplementedError


class Retryable(Exception):
    def __init__(self, result): self.result = result


async def call_with_retries(call, tool, args, run_id, keyed, read_only):
    # `call(args)` calls the tool and returns its result. `keyed`: the tool accepts an idempotency_key.
    # TODO 2: a. if `keyed`, add args["idempotency_key"], computed ONCE, before the first attempt
    #         b. wrap `await call(args)` in an inner function decorated with tenacity's @retry:
    #            retry_if_exception_type(Retryable), stop_after_attempt(3), wait_random_exponential(multiplier=0.2, max=2),
    #            retry_error_callback=lambda state: state.outcome.exception().result
    #         c. raise Retryable(result) only if typed_error(result) says "retryable" AND (read_only or keyed);
    #            otherwise return result: the model reads the error
    raise NotImplementedError


In [ ]:
await check(call_with_retries)             # no model: the same write twice must leave one change

## 4. Watch the agent again (v0.2)

Same task, same chaos, your server and your dispatcher. Why must the **harness**, not the model, compute the key? What did the agent see when a write timed out, before and after?

In [ ]:
after = await trials(call_with_retries)
after